In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # ARC LAB — Silver → Gold Layer
# MAGIC **Pipeline:** WattTime Carbon Intensity | CAISO_NORTH  
# MAGIC **Purpose:** Hourly aggregates + scheduling opportunity metrics for AI workload carbon optimization  
# MAGIC **Research question:** When and where is the grid clean enough to run AI workloads efficiently?

# COMMAND ----------

# MAGIC %md
# MAGIC ## Setup

# COMMAND ----------

from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOG   = "bootcamp_students"
SILVER_DB = "silver"
GOLD_DB   = "gold"
SILVER_TABLE = f"{CATALOG}.{SILVER_DB}.watttime_clean"
GOLD_TABLE   = f"{CATALOG}.{GOLD_DB}.carbon_intensity_hourly"

# COMMAND ----------

# MAGIC %md
# MAGIC ## 1. Create Gold Schema (idempotent)

# COMMAND ----------

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{GOLD_DB}")
print(f"Schema ready: {CATALOG}.{GOLD_DB}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 2. Load Silver

# COMMAND ----------

silver = spark.table(SILVER_TABLE)
print(f"Silver records loaded: {silver.count():,}")
silver.printSchema()

# COMMAND ----------

# Quick sanity check
silver.select(
    F.min("lbs_co2_per_mwh").alias("min_intensity"),
    F.max("lbs_co2_per_mwh").alias("max_intensity"),
    F.round(F.avg("lbs_co2_per_mwh"), 2).alias("avg_intensity"),
    F.countDistinct("region_code").alias("regions"),
    F.min("ts_utc").alias("earliest"),
    F.max("ts_utc").alias("latest")
).show(truncate=False)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 3. Feature Engineering
# MAGIC Extract temporal dimensions from ts_utc for aggregation

# COMMAND ----------

# NOTE: ts_utc is TIMESTAMP_NTZ — cast explicitly before extracting parts
silver_enriched = silver.withColumn(
    "ts_cast", F.col("ts_utc").cast("timestamp")
).withColumn(
    "hour_of_day",   F.hour("ts_cast")                      # 0–23
).withColumn(
    "day_of_week",   F.dayofweek("ts_cast")                 # 1=Sun, 7=Sat
).withColumn(
    "day_name",      F.date_format("ts_cast", "EEEE")       # Monday, Tuesday...
).withColumn(
    "is_weekend",    (F.dayofweek("ts_cast").isin([1, 7])).cast("boolean")
).withColumn(
    "date_local",    F.to_date("ts_cast")
).withColumn(
    "month",         F.month("ts_cast")
).withColumn(
    "year",          F.year("ts_cast")
)

print("Feature engineering complete.")
silver_enriched.select(
    "ts_utc", "hour_of_day", "day_name", "is_weekend", "lbs_co2_per_mwh"
).show(5, truncate=False)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 4. Hourly Aggregates
# MAGIC Average carbon intensity by region + hour_of_day across full dataset history.  
# MAGIC This is the core signal for scheduling recommendations.

# COMMAND ----------

hourly_agg = silver_enriched.groupBy(
    "region_code",
    "hour_of_day",
    "is_weekend"
).agg(
    F.round(F.avg("lbs_co2_per_mwh"), 2).alias("avg_intensity"),
    F.round(F.min("lbs_co2_per_mwh"), 2).alias("min_intensity"),
    F.round(F.max("lbs_co2_per_mwh"), 2).alias("max_intensity"),
    F.round(F.stddev("lbs_co2_per_mwh"), 2).alias("stddev_intensity"),
    F.round(
        F.expr("percentile_approx(lbs_co2_per_mwh, 0.25)"), 2
    ).alias("p25_intensity"),
    F.round(
        F.expr("percentile_approx(lbs_co2_per_mwh, 0.50)"), 2
    ).alias("p50_intensity"),
    F.round(
        F.expr("percentile_approx(lbs_co2_per_mwh, 0.75)"), 2
    ).alias("p75_intensity"),
    F.count("lbs_co2_per_mwh").alias("sample_count")
)

print(f"Hourly aggregate rows: {hourly_agg.count()}")
hourly_agg.orderBy("region_code", "is_weekend", "hour_of_day").show(10, truncate=False)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 5. Scheduling Opportunity Metrics
# MAGIC 
# MAGIC **Low-carbon window:** hour where avg_intensity <= 20th percentile for that region+weekend combo  
# MAGIC **High-carbon window:** hour where avg_intensity >= 80th percentile  
# MAGIC **Avoidable carbon delta:** difference between worst and best window — the OSV/DOE headline number

# COMMAND ----------

# Compute per-region percentile thresholds (weekday and weekend separately)
region_thresholds = hourly_agg.groupBy("region_code", "is_weekend").agg(
    F.round(
        F.expr("percentile_approx(avg_intensity, 0.20)"), 2
    ).alias("threshold_p20"),
    F.round(
        F.expr("percentile_approx(avg_intensity, 0.80)"), 2
    ).alias("threshold_p80"),
    F.round(F.min("avg_intensity"), 2).alias("region_min_avg"),
    F.round(F.max("avg_intensity"), 2).alias("region_max_avg")
)

# Join thresholds back to hourly aggregates
gold = hourly_agg.join(
    region_thresholds,
    on=["region_code", "is_weekend"],
    how="left"
).withColumn(
    # Flag low-carbon windows (best times to run AI workloads)
    "is_low_carbon_window",
    (F.col("avg_intensity") <= F.col("threshold_p20")).cast("boolean")
).withColumn(
    # Flag high-carbon windows (worst times)
    "is_high_carbon_window",
    (F.col("avg_intensity") >= F.col("threshold_p80")).cast("boolean")
).withColumn(
    # Carbon saved per MWh by shifting from worst to best window
    "avoidable_lbs_co2_per_mwh",
    F.round(F.col("region_max_avg") - F.col("region_min_avg"), 2)
).withColumn(
    # Scheduling opportunity score: 0–100, higher = more benefit from scheduling
    # = (max_avg - avg_intensity) / (max_avg - min_avg) * 100
    "scheduling_opportunity_score",
    F.round(
        F.when(
            (F.col("region_max_avg") - F.col("region_min_avg")) > 0,
            (F.col("region_max_avg") - F.col("avg_intensity"))
            / (F.col("region_max_avg") - F.col("region_min_avg"))
            * 100
        ).otherwise(0),
        1
    )
).withColumn(
    # Metadata
    "pipeline_version", F.lit("1.0")
).withColumn(
    "updated_at", F.current_timestamp()
)

print(f"Gold layer rows: {gold.count()}")
gold.orderBy("region_code", "is_weekend", "scheduling_opportunity_score".desc() if False else F.col("scheduling_opportunity_score").desc()).show(10, truncate=False)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 6. Spot-Check: Best and Worst Windows

# COMMAND ----------

print("=== TOP 10 BEST HOURS TO RUN AI WORKLOADS (CAISO_NORTH) ===")
gold.filter(F.col("is_low_carbon_window") == True) \
    .orderBy("avg_intensity") \
    .select(
        "region_code", "hour_of_day", "is_weekend",
        "avg_intensity", "scheduling_opportunity_score",
        "avoidable_lbs_co2_per_mwh"
    ).show(10, truncate=False)

print("=== TOP 10 WORST HOURS TO RUN AI WORKLOADS (CAISO_NORTH) ===")
gold.filter(F.col("is_high_carbon_window") == True) \
    .orderBy(F.col("avg_intensity").desc()) \
    .select(
        "region_code", "hour_of_day", "is_weekend",
        "avg_intensity", "scheduling_opportunity_score",
        "avoidable_lbs_co2_per_mwh"
    ).show(10, truncate=False)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 7. Write Gold Table (overwrite)

# COMMAND ----------

gold.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(GOLD_TABLE)

print(f"Gold table written: {GOLD_TABLE}")
print(f"Row count: {spark.table(GOLD_TABLE).count():,}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 8. ZORDER Optimization
# MAGIC Optimizes file layout for fast filtering by region + weekend flag (primary dashboard query pattern)

# COMMAND ----------

spark.sql(f"""
    OPTIMIZE {GOLD_TABLE}
    ZORDER BY (region_code, is_weekend, hour_of_day)
""")
print("ZORDER complete.")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 9. Final Validation

# COMMAND ----------

final = spark.table(GOLD_TABLE)

print("=== GOLD TABLE SUMMARY ===")
final.select(
    F.count("*").alias("total_rows"),
    F.countDistinct("region_code").alias("regions"),
    F.sum(F.col("is_low_carbon_window").cast("int")).alias("low_carbon_windows"),
    F.sum(F.col("is_high_carbon_window").cast("int")).alias("high_carbon_windows"),
    F.round(F.avg("avoidable_lbs_co2_per_mwh"), 2).alias("avg_avoidable_lbs_per_mwh"),
    F.round(F.max("avoidable_lbs_co2_per_mwh"), 2).alias("max_avoidable_lbs_per_mwh")
).show(truncate=False)

print("\n=== SCHEMA ===")
final.printSchema()

print(f"\n✅ Gold layer complete: {GOLD_TABLE}")
print("   Ready for: AI/BI Dashboard + Genie space")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Gold Layer Data Dictionary
# MAGIC 
# MAGIC | Column | Type | Description |
# MAGIC |---|---|---|
# MAGIC | region_code | string | Grid balancing area (e.g. CAISO_NORTH) |
# MAGIC | hour_of_day | int | Hour 0–23 UTC |
# MAGIC | is_weekend | boolean | True = Saturday/Sunday |
# MAGIC | avg_intensity | double | Avg lbs CO2/MWh for this hour slot |
# MAGIC | min_intensity | double | Historical minimum |
# MAGIC | max_intensity | double | Historical maximum |
# MAGIC | stddev_intensity | double | Variability measure |
# MAGIC | p25/p50/p75_intensity | double | Percentile distribution |
# MAGIC | sample_count | long | Number of raw readings in this bucket |
# MAGIC | threshold_p20 | double | Region's 20th percentile avg — low-carbon cutoff |
# MAGIC | threshold_p80 | double | Region's 80th percentile avg — high-carbon cutoff |
# MAGIC | region_min_avg / region_max_avg | double | Region-level best/worst hourly averages |
# MAGIC | is_low_carbon_window | boolean | Best hours to schedule AI workloads |
# MAGIC | is_high_carbon_window | boolean | Worst hours — avoid if possible |
# MAGIC | avoidable_lbs_co2_per_mwh | double | Carbon saved per MWh by shifting to optimal window |
# MAGIC | scheduling_opportunity_score | double | 0–100 score: how much this hour benefits from scheduling |
# MAGIC | pipeline_version | string | For lineage tracking |
# MAGIC | updated_at | timestamp | Last pipeline run |

Schema ready: bootcamp_students.gold
Silver records loaded: 2,016
root
 |-- region_code: string (nullable = true)
 |-- signal_type: string (nullable = true)
 |-- ts_utc: timestamp_ntz (nullable = true)
 |-- ts_local: timestamp_ntz (nullable = true)
 |-- ts_date: date (nullable = true)
 |-- ts_5min_bucket: timestamp_ntz (nullable = true)
 |-- lbs_co2_per_mwh: double (nullable = true)
 |-- model_version: string (nullable = true)
 |-- model_transition_flag: boolean (nullable = true)
 |-- range_flag: boolean (nullable = true)
 |-- quality_flag: string (nullable = true)
 |-- ingested_at: timestamp_ntz (nullable = true)
 |-- processed_at: timestamp_ntz (nullable = true)

+-------------+-------------+-------------+-------+-------------------+-------------------+
|min_intensity|max_intensity|avg_intensity|regions|earliest           |latest             |
+-------------+-------------+-------------+-------+-------------------+-------------------+
|35.0         |1089.0       |721.31       |1      

In [0]:
display(spark.table("bootcamp_students.gold.carbon_intensity_hourly").limit(5))

region_code,is_weekend,hour_of_day,avg_intensity,min_intensity,max_intensity,stddev_intensity,p25_intensity,p50_intensity,p75_intensity,sample_count,threshold_p20,threshold_p80,region_min_avg,region_max_avg,is_low_carbon_window,is_high_carbon_window,avoidable_lbs_co2_per_mwh,scheduling_opportunity_score,pipeline_version,updated_at
CAISO_NORTH,true,8,865.96,730.0,1008.0,131.57,736.0,747.0,992.0,24,520.42,1021.71,92.0,1025.75,false,false,933.75,17.1,1.0,2026-03-08T05:28:57.516Z
CAISO_NORTH,true,0,792.71,52.0,1032.0,389.49,215.0,1005.0,1015.0,24,520.42,1021.71,92.0,1025.75,false,false,933.75,25.0,1.0,2026-03-08T05:28:57.516Z
CAISO_NORTH,true,18,330.83,50.0,857.0,329.95,104.0,125.0,692.0,24,520.42,1021.71,92.0,1025.75,true,false,933.75,74.4,1.0,2026-03-08T05:28:57.516Z
CAISO_NORTH,true,10,1018.17,1011.0,1027.0,4.42,1015.0,1017.0,1021.0,24,520.42,1021.71,92.0,1025.75,false,false,933.75,0.8,1.0,2026-03-08T05:28:57.516Z
CAISO_NORTH,true,16,359.13,50.0,1082.0,387.6,52.0,111.0,658.0,24,520.42,1021.71,92.0,1025.75,true,false,933.75,71.4,1.0,2026-03-08T05:28:57.516Z


In [0]:
spark.table("bootcamp_students.gold.carbon_intensity_hourly").printSchema()

root
 |-- region_code: string (nullable = true)
 |-- is_weekend: boolean (nullable = true)
 |-- hour_of_day: integer (nullable = true)
 |-- avg_intensity: double (nullable = true)
 |-- min_intensity: double (nullable = true)
 |-- max_intensity: double (nullable = true)
 |-- stddev_intensity: double (nullable = true)
 |-- p25_intensity: double (nullable = true)
 |-- p50_intensity: double (nullable = true)
 |-- p75_intensity: double (nullable = true)
 |-- sample_count: long (nullable = true)
 |-- threshold_p20: double (nullable = true)
 |-- threshold_p80: double (nullable = true)
 |-- region_min_avg: double (nullable = true)
 |-- region_max_avg: double (nullable = true)
 |-- is_low_carbon_window: boolean (nullable = true)
 |-- is_high_carbon_window: boolean (nullable = true)
 |-- avoidable_lbs_co2_per_mwh: double (nullable = true)
 |-- scheduling_opportunity_score: double (nullable = true)
 |-- pipeline_version: string (nullable = true)
 |-- updated_at: timestamp (nullable = true)

